The concept
1. What a decision tree does:
     - A decision tree partitions the feature space into rectangular regions by asking a sequence of yes/no questions.

    Split criteria
        Gini(S) = 1 - sum(p_k^2)

        - where p_k is the proportion of class k in set S.

    Entropy
    measures the information content (disorder) in a node. Covered in Phase 1 Lesson 09.
        Entropy(S) = -sum(p_k * log2(p_k))

    Information gain is the reduction in imputrity(entropy or Gini) after a split

Stopping conditions

    1. Pre-pruning stops the tree before it fully grows:
        Maximum depth: stop splitting when the tree reaches a set depth
        Minimum samples per leaf: stop if a node has fewer than k samples
        Minimum information gain: stop if the best split improves impurity by less than a threshold
        Maximum leaf nodes: limit the total number of leaves\
        
    2. Post-pruning grows the full tree, then trims it back:
        Cost-complexity pruning (used by scikit-learn): adds a penalty proportional to the number of leaves. Increase the penalty to get smaller trees
        Reduced error pruning: remove a subtree if the validation error does not increase
        

Random forests: the power of ensembles

    Two sources of randomness make the trees diverse:
    
        Bagging (bootstrap aggregating)
        Feature randomization

Feature importance

    Mean Decrease in Impurity (MDI):
        importance(feature_j) = sum over all nodes where feature_j is used:
    (n_samples_at_node / n_total_samples) * impurity_decrease
    
        

In [16]:
class Node:
    def __init__(
        self,
        feature=None,
        threshold=None,
        left=None,
        right=None,
        value=None
    ):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value


class DecisionTree:
    def __init__(
        self,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        criterion="gini",
        max_features=None
    ):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.criterion = criterion
        self.max_features = max_features

        self.tree = None
        self.feature_importances_ = None

    def fit(self, X, y):
        self.n_features = len(X[0])
        self.feature_importances_ = [0.0] * self.n_features

        self.tree = self._build(X, y, depth=0)

        total = sum(self.feature_importances_)
        if total > 0:
            self.feature_importances_ = [
                x / total for x in self.feature_importances_
            ]

    def predict(self, X):
        return [self._predict_one(x, self.tree) for x in X]

    def _build(self, X, y, depth):

        # stopping conditions
        if len(set(y)) == 1:
            return Node(value=y[0])

        if self.max_depth is not None and depth >= self.max_depth:
            return Node(value=self._majority_class(y))

        if len(X) < self.min_samples_split:
            return Node(value=self._majority_class(y))

        # find best split
        best_feature, best_threshold = self._best_split(X, y)

        if best_feature is None:
            return Node(value=self._majority_class(y))

        X_left, y_left = [], []
        X_right, y_right = [], []

        for xi, yi in zip(X, y):
            if xi[best_feature] <= best_threshold:
                X_left.append(xi)
                y_left.append(yi)
            else:
                X_right.append(xi)
                y_right.append(yi)

        if (
            len(X_left) < self.min_samples_leaf
            or len(X_right) < self.min_samples_leaf
        ):
            return Node(value=self._majority_class(y))

        left = self._build(X_left, y_left, depth + 1)
        right = self._build(X_right, y_right, depth + 1)

        return Node(
            feature=best_feature,
            threshold=best_threshold,
            left=left,
            right=right
        )

    def _predict_one(self, x, node):

        if node.value is not None:
            return node.value

        if x[node.feature] <= node.threshold:
            return self._predict_one(x, node.left)

        return self._predict_one(x, node.right)

    def _majority_class(self, y):
        return max(set(y), key=y.count)
    def _gini(self, y):
        n = len(y)
    
        if n == 0:
            return 0
    
        impurity = 1.0
    
        for cls in set(y):
            p = y.count(cls) / n
            impurity -= p ** 2
    
        return impurity

    def _best_split(self, X, y):

        best_gain = -1
        best_feature = None
        best_threshold = None
    
        parent_gini = self._gini(y)
    
        n_features = len(X[0])
    
        for feature in range(n_features):
    
            values = sorted(set(row[feature] for row in X))
    
            for threshold in values:
    
                y_left = []
                y_right = []
    
                for xi, yi in zip(X, y):
    
                    if xi[feature] <= threshold:
                        y_left.append(yi)
                    else:
                        y_right.append(yi)
    
                if len(y_left) == 0 or len(y_right) == 0:
                    continue
    
                n = len(y)
    
                child_gini = (
                    len(y_left)/n * self._gini(y_left)
                    +
                    len(y_right)/n * self._gini(y_right)
                )
    
                gain = parent_gini - child_gini
    
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold
    
        return best_feature, best_threshold

Test it!!

In [17]:
X = [
    [0, 0],  # Rainy, Cool
    [0, 1],  # Rainy, Hot
    [1, 0],  # Sunny, Cool
    [1, 1],  # Sunny, Hot
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1],
]

y = [
    0,
    0,
    1,
    1,
    0,
    0,
    1,
    1,
]
tree = DecisionTree(max_depth=3)
tree.fit(X, y)

pred = tree.predict(X)
print(pred)

[0, 0, 1, 1, 0, 0, 1, 1]


In [18]:
import numpy as np
from collections import Counter

class Node:
    """A helper class representing a single point/split in a decision tree."""
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature          # Index of the feature we split on
        self.threshold = threshold      # Value we split at (e.g., feature_value < threshold)
        self.left = left                # Left child node
        self.right = right              # Right child node
        self.value = value              # If it's a leaf node, this holds the predicted class

    def is_leaf(self):
        return self.value is not None


class DecisionTree:
    """A basic Decision Tree Classifier optimized for our Random Forest."""
    def __init__(self, max_depth=10, min_samples_split=2, n_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_features = n_features    # Used for feature subsampling in Random Forest
        self.root = None

    def fit(self, X, y):
        # If n_features isn't set, default to using all features
        self.n_features = X.shape[1] if not self.n_features else min(X.shape[1], self.n_features)
        self.root = self._grow_tree(X, y)

    def _grow_tree(self, X, y, depth=0):
        n_samples, n_feats = X.shape
        n_labels = len(np.unique(y))

        # Base cases: Check stopping criteria
        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            most_common_label = Counter(y).most_common(1)[0][0]
            return Node(value=most_common_label)

        # Randomly select a subset of feature indices (Feature Subsampling)
        feat_idxs = np.random.choice(n_feats, self.n_features, replace=False)

        # Find the best split point
        best_feat, best_thresh = self._best_split(X, y, feat_idxs)

        # Create child splits
        left_idxs = np.argwhere(X[:, best_feat] <= best_thresh).flatten()
        right_idxs = np.argwhere(X[:, best_feat] > best_thresh).flatten()
        
        left_child = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right_child = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        
        return Node(feature=best_feat, threshold=best_thresh, left=left_child, right=right_child)

    def _best_split(self, X, y, feat_idxs):
        best_gain = -1
        split_idx, split_thresh = None, None

        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)
            
            for thresh in thresholds:
                gain = self._information_gain(y, X_column, thresh)
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_thresh = thresh
                    
        return split_idx, split_thresh

    def _information_gain(self, y, X_column, thresh):
        # Parent Gini
        parent_gini = self._gini(y)

        # Generate split indices
        left_idxs = np.argwhere(X_column <= thresh).flatten()
        right_idxs = np.argwhere(X_column > thresh).flatten()

        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0

        # Weighted average Gini of children
        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        gini_l, gini_r = self._gini(y[left_idxs]), self._gini(y[right_idxs])
        child_gini = (n_l / n) * gini_l + (n_r / n) * gini_r

        # Information Gain is the reduction in impurity
        return parent_gini - child_gini

    def _gini(self, y):
        proportions = np.bincount(y) / len(y)
        return 1.0 - np.sum([p**2 for p in proportions if p > 0])

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.is_leaf():
            return node.value
        
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)


class RandomForestClassifier:
    """The Random Forest Ensemble."""
    def __init__(self, n_trees=10, max_depth=10, min_samples_split=2, n_features=None):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_features = n_features
        self.trees = []

    def fit(self, X, y):
        self.trees = []
        for _ in range(self.n_trees):
            # 1. Create a Decision Tree instance
            tree = DecisionTree(
                max_depth=self.max_depth, 
                min_samples_split=self.min_samples_split, 
                n_features=self.n_features
            )
            # 2. Get a bootstrap sample of the training data
            X_sample, y_sample = self._bootstrap_samples(X, y)
            # 3. Fit the tree on this unique sample slice
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def _bootstrap_samples(self, X, y):
        n_samples = X.shape[0]
        # Randomly choose indices with replacement
        idxs = np.random.choice(n_samples, n_samples, replace=True)
        return X[idxs], y[idxs]

    def predict(self, X):
        # Gather predictions from all trees
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        # Transpose to group predictions by sample row instead of by tree
        tree_preds = np.swapaxes(tree_preds, 0, 1)
        
        # Take a majority vote for each data row
        final_preds = [Counter(sample_preds).most_common(1)[0][0] for sample_preds in tree_preds]
        return np.array(final_preds)